# 04. Schema Mapping Graph 테스트

**목적**: `app/graphs/schema_graph.py`의 스키마 매핑 파이프라인 동작 확인

**그래프 흐름**
```
START → retriever (임베딩 유사도 → 후보 3개) → reasoner (SLM JSON 추론) → END
```

**타겟 스키마**: user_id, user_name, phone_number, email_address, signup_date, last_login, is_active, shipping_address

**체크리스트**
- [ ] 임베딩 유사도 기반 후보 선정 확인
- [ ] SLM JSON 추론 결과 확인
- [ ] 다양한 소스 컬럼 매핑 정확도 확인
- [ ] 배치 매핑 테스트

In [ ]:
import sys
sys.path.insert(0, '..')

## 1. 그래프 구축

In [ ]:
from app.graphs.schema_graph import build_schema_graph

graph = build_schema_graph()
print("스키마 매핑 그래프 구축 완료")

## 2. 그래프 구조 시각화

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## 3. 단일 컬럼 매핑 테스트

In [ ]:
result = graph.invoke({
    "source_col": "m_hp",
    "sample_value": "010-1234-5678",
})

print(f"소스 컬럼   : m_hp")
print(f"후보 컬럼   : {result['candidates']}")
print(f"매핑 결과   : {result['final_mapping']}")
print(f"매핑 이유   : {result['reasoning']}")

## 4. 배치 매핑 테스트

In [ ]:
import pandas as pd

test_cases = [
    {"source_col": "m_hp",        "sample_value": "010-1234-5678"},
    {"source_col": "usr_nm",      "sample_value": "홍길동"},
    {"source_col": "mail",        "sample_value": "user@example.com"},
    {"source_col": "reg_dt",      "sample_value": "2024-01-15"},
    {"source_col": "addr",        "sample_value": "서울시 강남구"},
    {"source_col": "use_yn",      "sample_value": "Y"},
]

rows = []
for case in test_cases:
    result = graph.invoke(case)
    rows.append({
        "소스 컬럼": case["source_col"],
        "샘플 값": case["sample_value"],
        "후보": result["candidates"],
        "매핑 결과": result["final_mapping"],
        "이유": result["reasoning"],
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 60)
df